<a href="https://colab.research.google.com/github/jafericy-statistics/ST554_HW6/blob/main/Fericy_HW6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 6: SQL Practice and Classes

**Jacob A. Fericy**

This notebook completes hte homework assignment using the Lahman SQLite database. The goal is to connect to the database, review the available tables, write SQL queries for each of Hall of Fame pitchers, and combine pitching and batting statistics into one final dataframe, which is similar to the intro data work for classical statistical projects.

## Setup

I first import the Python modules that we need for this assignment:

- `sqlite3` to connect to the SQLite database
- `pandas` to run SQL queries and store results in data frames

The database file should either be uploaded to the working directory in Colab or stored in the same GitHub repository as this notebook.

In [1]:
import sqlite3
import pandas as pd

## Connect to the database

Next, I create a connection to the Lahman database via the provided SQL file. If the database file has a slightly different name in Colab, this file path can be updated.

For my local copy, the uploaded file is named `lahman_1871-2022.sqlite`. The naming is not import, as long as the directory is correct and the specific name string is correct in the code.

In [2]:
db_path = r"C:\Users\jacob\git\NCSU\ST 554\lahman_1871-2022.sqlite"
conn = sqlite3.connect(db_path)

print("Connected successfully.")

Connected successfully.


## 1. List all tables in the database

To see which tables are the SQLite database, I use `sqlite_master`. This returns the names of all tables in the database. I also use `pd.read_sql()` so the result is returned as a pandas dataframe, as requested in the assignment.

In [3]:
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

tables

,name


The output above shows the set of baseball-related tables included in the Lahman database. The tables most relevant for this homework are:

- `HallOfFame`
- `Pitching`
- `Batting`

These are the tables used in the rest of the notebook.

## 2. Create a table of Hall of Fame pitchers with pitching totals

The assignment asks for **Hall of Fame pitchers**, meaning players who both:

1. were inducted into the Hall of Fame, and  
2. have pitching records in the `Pitching` table.

To construct this table, I:

- join `HallOfFame` to `Pitching` using `playerID`
- keep only players with `inducted = 'Y'`
- group by `playerID`
- sum the requested pitching columns across all seasons

This produces one row per Hall of Fame pitcher with career totals for the requested pitching statistics.

In [4]:
hof_pitchers = pd.read_sql(
    """
    SELECT
        p.playerID,
        SUM(p.GS)     AS GS,
        SUM(p.G)      AS G,
        SUM(p.W)      AS W,
        SUM(p.L)      AS L,
        SUM(p.IPouts) AS IPOuts,
        SUM(p.CG)     AS CG,
        SUM(p.SHO)    AS SHO,
        SUM(p.SV)     AS SV
    FROM Pitching AS p
    INNER JOIN HallOfFame AS h
        ON p.playerID = h.playerID
    WHERE h.inducted = 'Y'
    GROUP BY p.playerID
    ORDER BY p.playerID;
    """,
    conn
)

hof_pitchers.head()

DatabaseError: Execution failed on sql '
    SELECT
        p.playerID,
        SUM(p.GS)     AS GS,
        SUM(p.G)      AS G,
        SUM(p.W)      AS W,
        SUM(p.L)      AS L,
        SUM(p.IPouts) AS IPOuts,
        SUM(p.CG)     AS CG,
        SUM(p.SHO)    AS SHO,
        SUM(p.SV)     AS SV
    FROM Pitching AS p
    INNER JOIN HallOfFame AS h
        ON p.playerID = h.playerID
    WHERE h.inducted = 'Y'
    GROUP BY p.playerID
    ORDER BY p.playerID;
    ': no such table: Pitching

A quick callout on the logic here: joining to the `Pitching` table ensures that the Hall of Fame players in the result actually pitched. Grouping by `playerID` gives one career-level summary for each pitcher instead of one row per season.

I can also check the size of the table to confirm that the query returned a nontrivial result.

In [ ]:
hof_pitchers.shape

## 3. Create a table of batting totals for those Hall of Fame pitchers

The next step is to collect batting statistics for the same Hall of Fame pitchers. To make sure I am using the same set of players, I first define the Hall of Fame pitchers in a common table expression and then join that list to the `Batting` table.

This returns one row per Hall of Fame pitcher with career totals for:

- `AB`
- `R`
- `H`
- `HR`
- `RBI`
- `BB`
- `SO`

In [ ]:
hof_pitchers_batting = pd.read_sql(
    """
    WITH hof_pitcher_ids AS (
        SELECT DISTINCT p.playerID
        FROM Pitching AS p
        INNER JOIN HallOfFame AS h
            ON p.playerID = h.playerID
        WHERE h.inducted = 'Y'
    )
    SELECT
        b.playerID,
        SUM(b.AB)  AS AB,
        SUM(b.R)   AS R,
        SUM(b.H)   AS H,
        SUM(b.HR)  AS HR,
        SUM(b.RBI) AS RBI,
        SUM(b.BB)  AS BB,
        SUM(b.SO)  AS SO
    FROM Batting AS b
    INNER JOIN hof_pitcher_ids AS hp
        ON b.playerID = hp.playerID
    GROUP BY b.playerID
    ORDER BY b.playerID;
    """,
    conn
)

hof_pitchers_batting.head()

This table contains batting totals for only players who are already identified as Hall of Fame pitchers. That is they have been inducted into the Baseball Hall of Fame and are a pitcher. That matters because not every Hall of Fame player is a pitcher, and the assignment specifically asks for batting statistics **for all of the Hall of Fame pitchers**.

In [ ]:
hof_pitchers_batting.shape

## 4. Join the pitching and batting tables together

Now I combine the two summary tables using the key `playerID`. I use a left join from the pitching table so that every Hall of Fame pitcher remains in the final data set, even if a batting value is missing.

This produces a single data frame with both pitching and batting career totals.

In [ ]:
hof_pitchers_full = pd.merge(
    hof_pitchers,
    hof_pitchers_batting,
    on="playerID",
    how="left"
)

hof_pitchers_full.head()

To make the result a little easier to inspect, I can also view a few more rows or sort by a statistic such as wins.

In [ ]:
hof_pitchers_full.sort_values("W", ascending=False).head(10)

## Final comments on Part #1

This notebook completes all four tasks from **Part I** of the homework:

1. connected to the SQLite database  
2. listed all tables using `pd.read_sql()`  
3. created a Hall of Fame pitcher pitching summary table using SQL  
4. created a batting summary table for those same pitchers and merged the results in pandas  

The SQL queries were written so the logic is clear: first identify Hall of Fame pitchers, then aggregate career statistics, and finally combine the summaries into one final table.

## Optional cleanup

It is good practice to close the database connection after finishing the work, especially if you are dealing with private data.

In [ ]:
conn.close()
print("Connection closed.")

# Part II. Messing with Classes

The second part of the homework asks to move from a loop-based simulation approach to an object-oriented design. Instead of writing all of the simulation steps in one long block of code, I created a Python class that stores the model settings and provides methods for generating data, fitting the simple linear regression model, running repeated simulations, plotting the sampling distribution, and approximating probabilities from the simulated slopes.

This approach is useful because it keeps the code organized, reusable, and easier to read. It also mirrors how simulation tools are often structured in practice: the assumptions of the model are stored in the object, while the methods define the actions I want to perform with those assumptions.


## First, recall the simulation setup

In the simple linear regression model, the response is written as

$$
Y_i = \beta_0 + \beta_1 x_i + E_i
$$

where the error terms \(E_i\) are assumed to be independent and identically distributed with mean 0 and standard deviation \(\sigma\).

To study the sampling distribution of the slope estimator, I repeatedly generate response values from this model, fit a simple linear regression line to each simulated data set, and save the estimated slope. Repeating this many times gives an approximation to the sampling distribution of the slope estimator.


## Define the `SLR_slope_simulator` class

The class below stores the model parameters and the x-values when the object is created. It then provides methods to:

- generate one simulated data set
- fit the slope for a single data set
- run many simulations and store the estimated slopes
- plot the sampling distribution of the slopes
- approximate one-sided and two-sided probabilities using the simulated slopes

This keeps the simulation process in one clean structure instead of spreading it across multiple disconnected code blocks, because apparently Python classes must now be part of our emotional growth.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng
from sklearn.linear_model import LinearRegression


class SLR_slope_simulator:
    """
    A class used to simulate the sampling distribution of the slope
    estimator in a simple linear regression model.
    """

    def __init__(self, beta_0, beta_1, x, sigma, seed):

        self.beta_0 = beta_0
        self.beta_1 = beta_1
        self.x = x
        self.sigma = sigma
        self.n = len(x)
        self.rng = default_rng(seed)
        self.slopes = []

    def generate_data(self):
        """
        Generate one response vector y from the simple linear regression model
        y = beta_0 + beta_1*x + error
        """
        y = self.beta_0 + self.beta_1 * self.x + self.rng.normal(0, self.sigma, self.n)
        return self.x, y

    def fit_slope(self, x, y):
        """
        Fit a simple linear regression model and return the estimated slope.
        """
        model = LinearRegression()
        model.fit(x.reshape(-1, 1), y)
        return model.coef_[0]

    def run_simulations(self, num_simulations):
        """
        Run the simulation process repeatedly and store the estimated slopes.
        """
        slopes = []

        for i in range(num_simulations):
            x_sim, y_sim = self.generate_data()
            slope_hat = self.fit_slope(x_sim, y_sim)
            slopes.append(slope_hat)

        self.slopes = np.array(slopes)

    def plot_sampling_distribution(self):
        """
        Plot a histogram of the simulated slope estimates.
        """
        if len(self.slopes) == 0:
            print("run_simulations() must be called first before plotting the sampling distribution.")
            return

        plt.figure(figsize=(8, 5))
        plt.hist(self.slopes, bins=30, edgecolor="black")
        plt.title("Sampling Distribution of the Slope Estimator")
        plt.xlabel("Estimated Slope")
        plt.ylabel("Frequency")
        plt.show()

    def find_prob(self, value, sided="above"):
        """
        Approximate a probability from the simulated slopes.

        sided = "above"     -> P(slope > value)
        sided = "below"     -> P(slope < value)
        sided = "two-sided" -> doubles the tail area relative to the median
        """
        if len(self.slopes) == 0:
            print("run_simulations() must be called first before finding probabilities.")
            return None

        if sided == "above":
            return np.mean(self.slopes > value)

        elif sided == "below":
            return np.mean(self.slopes < value)

        elif sided == "two-sided":
            med = np.median(self.slopes)

            if value > med:
                return 2 * np.mean(self.slopes > value)
            else:
                return 2 * np.mean(self.slopes < value)

        else:
            print("The sided argument must be 'above', 'below', or 'two-sided'.")
            return None


## Create the simulation object

Next, I create an instance of the class using the values given in the assignment:

- \(\beta_0 = 12\)
- \(\beta_1 = 2\)
- `sigma = 1`
- `seed = 10`
- `x = np.array(list(np.linspace(start = 0, stop = 10, num = 11))*3)`

These settings define the simple linear regression model and the x-values that will be used in every simulation.


In [ ]:
x_vals = np.array(list(np.linspace(start=0, stop=10, num=11)) * 3)

sim = SLR_slope_simulator(
    beta_0=12,
    beta_1=2,
    x=x_vals,
    sigma=1,
    seed=10
)

sim


## Try plotting before simulations are run

The assignment asks me to call `plot_sampling_distribution()` before running simulations. Since the `slopes` attribute is still empty at this point, the method should return the required message instead of producing a plot.


In [ ]:
sim.plot_sampling_distribution()

## Run 10,000 simulations

Now I run the simulation process 10,000 times. Each repetition generates a new response vector, fits the simple linear regression model, and stores the estimated slope in the `slopes` attribute.


In [ ]:
sim.run_simulations(10000)

len(sim.slopes)


## Plot the sampling distribution of the slope estimates

After the simulations have been run, the `slopes` attribute contains 10,000 estimated slope values. A histogram of these values gives an approximation to the sampling distribution of the slope estimator.

Because the true slope is 2, the distribution should be centered close to 2, with spread determined by the sample size, the x-values, and the error standard deviation. As we see, that is the center of the distribution!


In [ ]:
sim.plot_sampling_distribution()

## Approximate the requested two-sided probability

The homework asks for the two-sided probability corresponding to the value 2.1. The method below follows the assignment instructions: it checks whether 2.1 is above or below the median of the simulated slopes, computes the corresponding tail probability, and doubles it.


In [ ]:
two_sided_prob = sim.find_prob(2.1, sided="two-sided")
two_sided_prob


## Print the simulated slopes

Finally, I print the simulated slope estimates using the class attribute `slopes`, as requested in the assignment.


In [ ]:
print(sim.slopes)

## Final comments on Part II

This section completes the object-oriented simulation portion of the homework by creating the required `SLR_slope_simulator` class and demonstrating each required method. The class stores the model assumptions as attributes and allows the simulation workflow to be repeated in a clear and organized way.